# E09：DataFrame 读取、检查与变量选择

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 Pandas 读取 CSV 格式的表格文本。
2. 检查 DataFrame 的行列、字段、类型、缺失概况和基础统计信息。
3. 根据分析目的选择单列和多列，并区分变量在表格中的作用。
4. 在清洗前形成可复核的数据质量审计结论。
5. 完成一次 Excel 写出和读回，并核验表格结构。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查行数、字段、主键和缺失值口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- DataFrame 结果异常时，先检查 `shape`、字段名、数据类型和缺失值。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 路径确认与读取数据：10 分钟
- DataFrame 结构检查：20 分钟
- 变量选择与变量角色：15 分钟
- 运行课程审计代码并解释结果：15 分钟
- Excel 往返：10 分钟
- 审计结论、调试与提交：20 分钟

## 主线输入：再次认识同一批调查记录

E09 使用 E05 已经处理过的同源调查数据，但这次直接用 Pandas 读取原始表格。课程准备单元只负责稳定定位文件；本实验先观察和记录问题，不提前修改原始 DataFrame。

In [ ]:
from pathlib import Path

import pandas as pd

chapter4_dir = Path(".").resolve()
notebook_names = ["e09.ipynb", "e09_inclass.ipynb"]
if not any((chapter4_dir / name).exists() for name in notebook_names):
    chapter4_dir = Path("lab/applied-stat-64/chapter4").resolve()

shared_dir = chapter4_dir.parents[1] / "shared"
raw_path = shared_dir / "raw" / "student_survey_raw.txt"
output_dir = chapter4_dir / "e09_output"
excel_path = output_dir / "student_survey_roundtrip.xlsx"

if not raw_path.exists():
    raise FileNotFoundError(f"找不到课程数据：{raw_path}")

print("输入文件：", raw_path)
print("输入文件存在：", raw_path.exists())

## 任务 1：读取并预览原始调查表

使用 `pd.read_csv()` 读取原始表格，输出前 5 行、行列数和字段名。先保留读取结果，不在这个任务中清洗字段。

In [ ]:
raw_df = ...

print(...)
print("行列数：", ...)
print("字段名：", ...)

## 任务 2：检查 DataFrame 结构

依次使用 `info()` 和 `describe()` 检查数据。重点观察两个自报数值字段为什么没有直接成为数值类型，以及不同统计函数使用的有效记录范围。

In [ ]:
print("DataFrame 类型：", ...)
print("\n结构信息：")
...

print("\n分类与数值概况：")
print(...)

## 任务 3：选择变量并说明角色

分别选择：

1. 标识变量 `student_id`；
2. 分类变量 `gender`、`major` 和 `support_plan`；
3. 数值候选变量 `reported_study_hours` 和 `reported_attendance_rate`。

检查单列结果和多列结果的类型与形状。

In [ ]:
student_ids = ...
category_columns = ...
numeric_candidates = ...

print("单列类型：", ...)
print("分类变量表 shape：", ...)
print("数值候选变量表 shape：", ...)
print(...)

## 任务 4：运行课程提供的数据质量审计

下面的审计代码会临时检查唯一学生数、重复标识、类别取值和数值转换问题。本单元中的字符串清理、数值转换和重复检查将在 E10 正式学习；本实验只要求运行代码、阅读输出并说明它发现了什么，不要求独立编写这些表达式。

审计代码不会修改 `raw_df`。

In [ ]:
clean_ids = raw_df["student_id"].astype("string").str.strip()
study_numeric_probe = pd.to_numeric(
    raw_df["reported_study_hours"],
    errors="coerce",
)
attendance_numeric_probe = pd.to_numeric(
    raw_df["reported_attendance_rate"]
    .astype("string")
    .str.strip()
    .str.replace("%", "", regex=False),
    errors="coerce",
)

audit_summary = pd.Series(
    {
        "raw_rows": len(raw_df),
        "columns": raw_df.shape[1],
        "unique_student_ids_after_strip": clean_ids.nunique(),
        "duplicate_student_ids_after_strip": clean_ids.duplicated().sum(),
        "study_hours_not_numeric": study_numeric_probe.isna().sum(),
        "attendance_not_numeric": attendance_numeric_probe.isna().sum(),
    },
    name="audit_value",
)

print(audit_summary)
for column in ["gender", "major", "prior_programming", "support_plan"]:
    values = (
        raw_df[column]
        .astype("string")
        .str.strip()
        .value_counts(dropna=False)
    )
    print(f"\n{column} 原始取值：")
    print(values)

## 任务 5：完成一次 Excel 写出和读回

把原始 DataFrame 写入个人输出目录中的 Excel 文件，再使用 `pd.read_excel()` 读回。核对行列数和字段顺序；这个文件只是格式往返练习，不是正式清洗结果。

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
...
excel_df = ...

print("Excel 输出：", excel_path)
print("行列数一致：", ...)
print("字段顺序一致：", ...)

## 任务 6：写出审计结论

根据课程审计代码的输出，用 Markdown 说明：

1. 哪些问题会影响主键、分类或数值计算；
2. 哪些现象只是显示或格式问题；
3. E10 至少需要实施哪些清洗规则。

重点是把输出翻译成清洗计划，不需要解释尚未学习的每个 API 细节。

### 数据审计结论

- ...
- ...
- ...
- ...

## 实验小结

请用 Markdown 回答：

1. 为什么读取表格后不能立刻开始计算均值？
2. `Series` 和 `DataFrame` 在变量选择时有什么区别？
3. 为什么应在清洗前记录原始行数、字段和问题？

### 实验小结

- ...
- ...
- ...